# Week 1 · Day 3 — DVC Init, Remote Storage & Data Pipeline Scaffold

---

## 🔁 Day 2 Recap

- MLflow SQLite tracking server configured ✅
- 5 named experiments created (`W1_Infrastructure`, `W3_SKU_Classification`, `W6_Demand_Forecasting`, `W8_GeoRisk_Newsvendor`, `W9_Supplier_Qualification`) ✅
- `MLproject` YAML + entry points written ✅
- Dummy experiment logged (params + metrics + artifacts) ✅
- Model Registry: first model version in `Staging` ✅
- MLflow UI verified at `localhost:5000` ✅

---

## 🎯 Day 3 Objective

Initialise **DVC (Data Version Control)** so every raw dataset, processed artefact, and external feed is tracked, reproducible, and shareable — independent of Git's file-size limits.

| Step | Action | Output |
|------|--------|--------|
| 1 | `dvc init` + validate `.dvc/` structure | DVC config committed to Git |
| 2 | Configure local remote (fallback, no cloud needed) | `dvc remote list` shows `localremote` |
| 3 | (Optional) Configure S3/GDrive remote | Cloud push/pull ready |
| 4 | Write `dvc.yaml` pipeline stages | `dvc repro` skeleton for all 12 weeks |
| 5 | Seed `data/` dirs with `.gitkeep` + `.dvcignore` | Clean data layer |
| 6 | Create `src/utils/data_io.py` — typed load/save helpers | Reusable I/O across all notebooks |
| 7 | Log DVC metadata to MLflow W1 experiment | Traceability from day one |
| 8 | `dvc status` green — commit Day 3 | `git log` shows W1D3 commit |

---

## ❓ Why DVC Before Any Data Work?

The pipeline touches **three external data sources** (M5 Walmart, UN Comtrade, SIPRI) whose raw files are 100 MB–2 GB.  
Git cannot store them. DVC solves this by:
- Storing only a small `.dvc` pointer in Git
- Pushing the actual bytes to a remote (local folder, S3, GDrive, Azure)
- Reproducing any pipeline stage from any commit with `dvc repro`

Without DVC from Day 3 → by Week 5 you have un-tracked 500 MB CSVs and no way to reproduce your W3 classifier.

---

## ⚙️ Architecture (end-state of Day 3)

```
geo-aware-mro/
├─ .dvc/
│   ├─ config          ← remote URLs, core settings
│   ├─ .gitignore      ← auto-generated by dvc init
│   └─ tmp/            ← DVC cache temp (gitignored)
├─ .dvcignore          ← patterns DVC should never track
├─ dvc.yaml            ← pipeline DAG (12 stages)
├─ dvc.lock            ← auto-generated on dvc repro (gittracked)
├─ data/
│   ├─ raw/            ← source data (DVC-tracked, not in Git)
│   ├─ processed/      ← cleaned / feature-engineered (DVC-tracked)
│   ├─ external/       ← UN Comtrade, SIPRI exports (DVC-tracked)
│   └─ interim/        ← staging scratch (DVC-tracked)
├─ src/
│   └─ utils/
│       └─ data_io.py  ← typed load/save helpers (new today)
```

---
## STEP 0 — Path setup & Day 2 gate

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# ── Resolve project root (same pattern as Day 1 & 2) ─────────────────────────
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    ROOT = cwd
    for parent in [cwd] + list(cwd.parents):
        if (parent / ".git").exists() or (parent / "environment.yml").exists():
            ROOT = parent
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"📁 Project root : {ROOT}")
print(f"📦 .git exists  : {(ROOT / '.git').exists()}")
print(f"📦 mlflow.db    : {list(ROOT.rglob('mlflow.db'))}")

# ── Day 2 gate: MLflow must already be configured ────────────────────────────
import mlflow
from src.config.settings import settings

mlflow.set_tracking_uri(settings.MLFLOW_URI)
exp = mlflow.get_experiment_by_name("W1_Infrastructure")
assert exp is not None, (
    "❌ MLflow experiment 'W1_Infrastructure' not found. "
    "Did you complete Day 2? Run W1_D2_mlflow_setup.ipynb first."
)
print(f"\n✅ MLflow W1_Infrastructure experiment confirmed (id={exp.experiment_id})")
print("✅ Day 2 gate passed — proceeding to Day 3")

---
## STEP 1 — DVC installation check & `dvc init`

DVC was installed in `environment.yml` on Day 1.  
This step verifies the version and initialises the `.dvc/` directory if it doesn't exist yet.

In [ ]:
import subprocess
import shutil

# ── 1a. Version check ─────────────────────────────────────────────────────────
dvc_path = shutil.which("dvc")
assert dvc_path, (
    "❌ `dvc` not found on PATH. "
    "Fix: conda activate geo-mro && pip install dvc>=3.30"
)

result = subprocess.run(["dvc", "--version"], capture_output=True, text=True)
dvc_version = result.stdout.strip()
print(f"✅ DVC version  : {dvc_version}")
print(f"   Binary path  : {dvc_path}")

major = int(dvc_version.split(".")[0])
assert major >= 3, f"❌ Need DVC ≥ 3.x, got {dvc_version}. Run: pip install --upgrade dvc"

# ── 1b. dvc init (idempotent) ────────────────────────────────────────────────
dvc_dir = ROOT / ".dvc"

if not dvc_dir.exists():
    print("\n→ Running dvc init...")
    r = subprocess.run(
        ["dvc", "init"],
        cwd=ROOT,
        capture_output=True,
        text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"dvc init failed:\n{r.stderr}")
    print("✅ dvc init complete")
else:
    print("⏩ .dvc/ already exists — skipping dvc init")

# ── 1c. Verify .dvc structure ────────────────────────────────────────────────
required = [".dvc/config", ".dvc/.gitignore"]
for f in required:
    p = ROOT / f
    status = "✅" if p.exists() else "❌ MISSING"
    print(f"  {status}  {f}")

---
## STEP 2 — Configure local remote

A **local remote** is just a directory outside the repo that acts as the DVC cache store.  
It requires zero cloud credentials — perfect for Day 3.  
Step 3 optionally layers an S3 or GDrive remote on top.

```
~/geo-mro-dvc-remote/     ← local remote (outside repo, not in Git)
geo-aware-mro/
├─ .dvc/config            ← stores the remote URL
```

In [ ]:
import os
from pathlib import Path

# ── Local remote path: sibling of the repo root ───────────────────────────────
LOCAL_REMOTE = Path.home() / "geo-mro-dvc-remote"
LOCAL_REMOTE.mkdir(parents=True, exist_ok=True)
print(f"→ Local remote path : {LOCAL_REMOTE}")

# ── Add remote (idempotent: remove first if it exists to avoid errors) ────────
def run_dvc(*args: str) -> str:
    """Run a dvc command from ROOT, return stdout, raise on failure."""
    r = subprocess.run(
        ["dvc", *args],
        cwd=ROOT,
        capture_output=True,
        text=True
    )
    if r.returncode != 0:
        # Ignore 'already exists' errors gracefully
        if "already exists" not in r.stderr.lower():
            raise RuntimeError(f"dvc {' '.join(args)} failed:\n{r.stderr}")
    return r.stdout.strip()

# Remove then re-add to stay idempotent
try:
    run_dvc("remote", "remove", "localremote")
except Exception:
    pass  # remote didn't exist yet — fine

run_dvc("remote", "add", "--default", "localremote", str(LOCAL_REMOTE))
print("✅ 'localremote' added as default DVC remote")

# ── Verify ────────────────────────────────────────────────────────────────────
remotes = run_dvc("remote", "list")
print(f"\nDVC remotes:\n{remotes}")

config_content = (ROOT / ".dvc" / "config").read_text()
print(f"\n.dvc/config:\n{config_content}")

---
## STEP 3 — (Optional) Cloud remote: S3 or Google Drive

Skip this step if you don't have cloud credentials today.  
The local remote from Step 2 is sufficient for solo development.

### Option A — AWS S3
```bash
pip install dvc-s3
dvc remote add s3remote s3://your-bucket/geo-mro-dvc
dvc remote modify s3remote region us-east-1
# Credentials via ~/.aws/credentials or env vars AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY
```

### Option B — Google Drive
```bash
pip install dvc-gdrive
dvc remote add gdriveremote gdrive://your-folder-id
# First push triggers OAuth browser flow
```

### Option C — Azure Blob
```bash
pip install dvc-azure
dvc remote add azureremote azure://your-container/geo-mro-dvc
dvc remote modify azureremote connection_string 'DefaultEndpointsProtocol=...'
```

In [ ]:
# ── Cloud remote configuration (edit and uncomment your preferred option) ─────
CLOUD_PROVIDER = None   # Set to: 's3' | 'gdrive' | 'azure' | None

if CLOUD_PROVIDER == 's3':
    S3_BUCKET  = "your-bucket"          # ← update
    S3_PREFIX  = "geo-mro-dvc"
    S3_REGION  = "us-east-1"            # ← update

    run_dvc("remote", "add", "s3remote", f"s3://{S3_BUCKET}/{S3_PREFIX}")
    run_dvc("remote", "modify", "s3remote", "region", S3_REGION)
    run_dvc("remote", "default", "s3remote")
    print(f"✅ S3 remote configured: s3://{S3_BUCKET}/{S3_PREFIX}")

elif CLOUD_PROVIDER == 'gdrive':
    GDRIVE_FOLDER_ID = "your-folder-id"  # ← update (from Drive URL)
    run_dvc("remote", "add", "gdriveremote", f"gdrive://{GDRIVE_FOLDER_ID}")
    run_dvc("remote", "default", "gdriveremote")
    print(f"✅ GDrive remote configured: gdrive://{GDRIVE_FOLDER_ID}")
    print("   NOTE: First `dvc push` will open a browser for OAuth.")

elif CLOUD_PROVIDER == 'azure':
    AZURE_CONTAINER = "your-container"   # ← update
    run_dvc("remote", "add", "azureremote", f"azure://{AZURE_CONTAINER}/geo-mro-dvc")
    # Set connection string via env var: AZURE_STORAGE_CONNECTION_STRING
    print(f"✅ Azure remote configured: azure://{AZURE_CONTAINER}/geo-mro-dvc")
else:
    print("⏩ CLOUD_PROVIDER=None — using local remote only (fine for today)")

print(f"\nActive default remote: {run_dvc('remote', 'default')}")

---
## STEP 4 — Write `.dvcignore`

Like `.gitignore` but for DVC — prevents it from hashing files that should never be tracked  
(MLflow runs, Jupyter checkpoints, Python caches, etc.).

In [ ]:
DVCIGNORE_CONTENT = """\
# DVC ignore file — patterns DVC should NEVER track
# Analogous to .gitignore but for DVC cache

# Python
__pycache__/
*.py[cod]
*.egg-info/

# Jupyter
.ipynb_checkpoints/
*.ipynb

# MLflow artefacts (managed by MLflow, not DVC)
mlflow/mlruns/
mlflow/mlflow.db

# Environment
.env
*.env

# OS
.DS_Store
Thumbs.db

# Logs
logs/
*.log

# Test artefacts
.pytest_cache/
htmlcov/
"""

dvcignore_path = ROOT / ".dvcignore"
dvcignore_path.write_text(DVCIGNORE_CONTENT)
print(f"✅ .dvcignore written to {dvcignore_path}")
print(f"   Lines: {len(DVCIGNORE_CONTENT.splitlines())}")

---
## STEP 5 — Seed `data/` directories with metadata stubs

Each `data/` subdirectory gets:
- A `README.md` describing what goes in it
- A `schema.json` stub (filled as data arrives in W2+)
- A `.gitkeep` (replaced by real `.dvc` files when data is added)

This ensures the directory structure is committed to Git even before any data exists.

In [ ]:
import json

DATA_LAYER_METADATA: dict[str, dict] = {
    "data/raw": {
        "readme": (
            "# data/raw\n\n"
            "**Immutable** source data. Never modified after ingestion.\n\n"
            "## Expected contents (added in W2)\n"
            "| File | Source | Week | Size (approx) |\n"
            "|------|--------|------|---------------|\n"
            "| `m5_sales_train_evaluation.csv` | Kaggle M5 | W2 | ~130 MB |\n"
            "| `m5_calendar.csv` | Kaggle M5 | W2 | ~50 KB |\n"
            "| `m5_sell_prices.csv` | Kaggle M5 | W2 | ~140 MB |\n"
            "| `comtrade_hs_codes.csv` | UN Comtrade API | W2 | ~5 MB |\n"
            "| `sipri_arms_transfers.csv` | SIPRI | W8 | ~2 MB |\n\n"
            "## DVC tracking\n"
            "All files here are tracked via DVC — not committed to Git directly.\n"
            "After adding a file: `dvc add data/raw/<filename> && git add data/raw/<filename>.dvc`\n"
        ),
        "schema": {
            "layer": "raw",
            "mutability": "immutable",
            "dvc_tracked": True,
            "files": []
        }
    },
    "data/processed": {
        "readme": (
            "# data/processed\n\n"
            "Cleaned, validated, feature-engineered data.\n"
            "Output of the `preprocess` DVC stage.\n\n"
            "## Expected contents\n"
            "| File | Produced by | Week |\n"
            "|------|-------------|------|\n"
            "| `sku_master_v1.0.parquet` | W3 classifier | W4 |\n"
            "| `sku_master_v1.1.parquet` | W11 integration | W11 |\n"
            "| `demand_features.parquet` | W5 ADI/CV² | W5 |\n"
            "| `forecasts_12w.parquet` | W6-W7 engines | W7 |\n"
            "| `rop_table.parquet` | W8 newsvendor | W8 |\n"
        ),
        "schema": {
            "layer": "processed",
            "mutability": "versioned",
            "dvc_tracked": True,
            "files": []
        }
    },
    "data/external": {
        "readme": (
            "# data/external\n\n"
            "Third-party reference data (UN Comtrade, SIPRI, World Bank, etc.)\n"
            "Downloaded via API or manual export.\n\n"
            "## Contents\n"
            "| File | Source | Used in |\n"
            "|------|--------|---------|\n"
            "| `comtrade_hs_import_data.parquet` | UN Comtrade REST API | W2, W8 |\n"
            "| `sipri_conflict_index.parquet` | SIPRI | W8 GeoRisk |\n"
            "| `wb_country_risk.parquet` | World Bank | W8 |\n"
            "| `hhi_concentration.parquet` | Computed from Comtrade | W2, W10 |\n"
        ),
        "schema": {
            "layer": "external",
            "mutability": "append-only",
            "dvc_tracked": True,
            "files": []
        }
    },
    "data/interim": {
        "readme": (
            "# data/interim\n\n"
            "Staging scratch area. Intermediate artefacts between raw → processed.\n"
            "OK to delete and regenerate via `dvc repro`.\n\n"
            "## Examples\n"
            "- `m5_duckdb_profile.json` — null/dtype report from W2 ingest\n"
            "- `sku_synthetic_500.parquet` — pre-join synthetic SKU master\n"
            "- `abc_pareto_raw.parquet` — ACV cumsum before cutoff assignment\n"
        ),
        "schema": {
            "layer": "interim",
            "mutability": "ephemeral",
            "dvc_tracked": False,
            "files": []
        }
    }
}

for dir_rel, meta in DATA_LAYER_METADATA.items():
    dir_path = ROOT / dir_rel
    dir_path.mkdir(parents=True, exist_ok=True)

    readme_p = dir_path / "README.md"
    readme_p.write_text(meta["readme"])

    schema_p = dir_path / "schema.json"
    schema_p.write_text(json.dumps(meta["schema"], indent=2))

    gitkeep_p = dir_path / ".gitkeep"
    gitkeep_p.touch(exist_ok=True)

    print(f"✅ {dir_rel}/ — README.md + schema.json + .gitkeep")

print("\n✅ data/ layer seeded")

---
## STEP 6 — Write `dvc.yaml` pipeline DAG

This is the **central nervous system** of the project's data pipeline.  
Each stage maps to a roadmap week. Right now the `cmd` fields are stubs —  
they will be replaced with real scripts as each week completes.

```
dvc repro          → runs ALL stages in dependency order
dvc repro preprocess  → runs only the preprocess stage + upstreams
dvc dag            → prints the ASCII pipeline graph
```

In [ ]:
DVC_YAML = """\
# dvc.yaml — Geo-Aware MRO · 12-Week Pipeline DAG
# Run: dvc repro
# Visualise: dvc dag
# ─────────────────────────────────────────────────────────────────────────────

stages:

  # ── W2: Data Ingestion ─────────────────────────────────────────────────────
  ingest_m5:
    cmd: python src/ingestion/ingest_m5.py
    deps:
      - src/ingestion/ingest_m5.py
    outs:
      - data/raw/m5_sales_train_evaluation.csv:
          cache: true
          persist: true
      - data/raw/m5_calendar.csv:
          cache: true
          persist: true
      - data/raw/m5_sell_prices.csv:
          cache: true
          persist: true
    params:
      - params.yaml:
          - ingestion.m5_version

  ingest_comtrade:
    cmd: python src/ingestion/ingest_comtrade.py
    deps:
      - src/ingestion/ingest_comtrade.py
    outs:
      - data/external/comtrade_hs_import_data.parquet:
          cache: true
    params:
      - params.yaml:
          - ingestion.hs_codes
          - ingestion.reporter_countries

  # ── W2: SKU Master Schema ──────────────────────────────────────────────────
  build_sku_synthetic:
    cmd: python src/ingestion/build_sku_synthetic.py
    deps:
      - src/ingestion/build_sku_synthetic.py
      - data/external/comtrade_hs_import_data.parquet
    outs:
      - data/interim/sku_synthetic_500.parquet:
          cache: true
    params:
      - params.yaml:
          - sku_master.n_skus
          - sku_master.random_seed

  # ── W3: ABC × VED × FNS Classification ────────────────────────────────────
  classify_sku:
    cmd: python src/classifiers/run_classification.py
    deps:
      - src/classifiers/abc_classifier.py
      - src/classifiers/ved_classifier.py
      - src/classifiers/fns_classifier.py
      - src/classifiers/run_classification.py
      - data/interim/sku_synthetic_500.parquet
    outs:
      - data/processed/sku_master_v1.0.parquet:
          cache: true
      - data/processed/classification_report.html:
          cache: true
    params:
      - params.yaml:
          - classifier.abc_pareto_a
          - classifier.abc_pareto_b
          - classifier.adi_threshold
          - classifier.cv2_threshold
          - classifier.w1
          - classifier.w2
          - classifier.w3
          - classifier.w4
    metrics:
      - data/processed/classification_metrics.json:
          cache: false

  # ── W5-W6: Demand Characterisation + Croston's / SBA ──────────────────────
  demand_characterise:
    cmd: python src/forecasting/demand_characterise.py
    deps:
      - src/forecasting/demand_characterise.py
      - data/raw/m5_sales_train_evaluation.csv
      - data/processed/sku_master_v1.0.parquet
    outs:
      - data/processed/demand_features.parquet:
          cache: true
    params:
      - params.yaml:
          - forecasting.adi_threshold
          - forecasting.cv2_threshold

  forecast_intermittent:
    cmd: python src/forecasting/run_croston_sba.py
    deps:
      - src/forecasting/run_croston_sba.py
      - src/forecasting/croston.py
      - data/processed/demand_features.parquet
    outs:
      - data/processed/forecasts_intermittent.parquet:
          cache: true
    params:
      - params.yaml:
          - forecasting.croston_alpha
          - forecasting.forecast_horizon_weeks

  # ── W7: Holt-Winters / ARIMA ───────────────────────────────────────────────
  forecast_fast_movers:
    cmd: python src/forecasting/run_hw_arima.py
    deps:
      - src/forecasting/run_hw_arima.py
      - data/processed/demand_features.parquet
    outs:
      - data/processed/forecasts_fast_movers.parquet:
          cache: true
    params:
      - params.yaml:
          - forecasting.arima_max_p
          - forecasting.arima_max_d
          - forecasting.arima_max_q
          - forecasting.forecast_horizon_weeks

  # ── W8: Bayesian Geo-Risk + Newsvendor ─────────────────────────────────────
  georisk_score:
    cmd: python src/risk/run_georisk.py
    deps:
      - src/risk/run_georisk.py
      - src/risk/bayesian_georisk.py
      - data/processed/sku_master_v1.0.parquet
      - data/external/comtrade_hs_import_data.parquet
      - data/external/sipri_conflict_index.parquet
    outs:
      - data/processed/sku_master_georisk.parquet:
          cache: true
    params:
      - params.yaml:
          - georisk.lambda_adjustment
          - georisk.prior_alpha
          - georisk.prior_beta

  newsvendor_baseline:
    cmd: python src/risk/run_newsvendor.py
    deps:
      - src/risk/run_newsvendor.py
      - data/processed/sku_master_georisk.parquet
      - data/processed/forecasts_intermittent.parquet
      - data/processed/forecasts_fast_movers.parquet
    outs:
      - data/processed/rop_table.parquet:
          cache: true
    params:
      - params.yaml:
          - newsvendor.service_level

  # ── W9-W10: Game Theory ─────────────────────────────────────────────────────
  supplier_qualify:
    cmd: python src/game_theory/run_supplier_qualify.py
    deps:
      - src/game_theory/run_supplier_qualify.py
      - data/processed/sku_master_georisk.parquet
    outs:
      - data/processed/sku_master_v1.1.parquet:
          cache: true
    params:
      - params.yaml:
          - game_theory.dt_max_depth
          - game_theory.nash_iterations
"""

dvc_yaml_path = ROOT / "dvc.yaml"
dvc_yaml_path.write_text(DVC_YAML)
print(f"✅ dvc.yaml written ({len(DVC_YAML.splitlines())} lines)")
print(f"   Stages defined: {DVC_YAML.count('cmd:')}")

# Validate the YAML is parseable
import yaml
with open(dvc_yaml_path) as f:
    parsed = yaml.safe_load(f)
print(f"   YAML parse OK  : {list(parsed['stages'].keys())}")

---
## STEP 7 — Write `params.yaml`

`params.yaml` is the **single source of truth for all hyperparameters** in the pipeline.  
DVC tracks changes to it — if you change `classifier.w1`, DVC knows to re-run `classify_sku`.  
MLflow also reads from this file so experiments are reproducible.

In [ ]:
PARAMS_YAML = """\
# params.yaml — All pipeline hyperparameters
# Tracked by DVC. Changes trigger stage re-runs.
# Read by MLflow experiments for full traceability.
# ─────────────────────────────────────────────────────────────────────────────

# ── Data Ingestion (W2) ───────────────────────────────────────────────────────
ingestion:
  m5_version: "evaluation"          # or 'validation' for public leaderboard set
  hs_codes: ["8803", "8804", "8805"] # Aircraft parts (HS Chapter 88)
  reporter_countries: ["356", "840", "276", "826"]  # India, USA, Germany, UK

# ── SKU Master Synthetic Generator (W2) ──────────────────────────────────────
sku_master:
  n_skus: 500
  random_seed: 42
  unit_cost_range: [100, 250000]     # INR
  lead_time_range_days: [7, 180]
  demand_mean_range: [0.1, 50.0]     # units/week

# ── ABC × VED × FNS Classifier (W3) ──────────────────────────────────────────
classifier:
  # Pareto cutoffs (cumulative ACV %)
  abc_pareto_a: 0.70                 # top 70% ACV → Class A
  abc_pareto_b: 0.90                 # 70–90% ACV → Class B
  # Syntetos-Boylan demand classification boundaries
  adi_threshold: 1.32
  cv2_threshold: 0.49
  # Composite criticality index weights (must sum to 1.0)
  w1: 0.35  # ABC weight
  w2: 0.30  # VED weight
  w3: 0.20  # FNS weight
  w4: 0.15  # Location weight

# ── Demand Forecasting (W5-W7) ────────────────────────────────────────────────
forecasting:
  adi_threshold: 1.32
  cv2_threshold: 0.49
  croston_alpha: 0.1                 # ETS smoothing param (sizes)
  croston_alpha_p: 0.1               # ETS smoothing param (intervals)
  forecast_horizon_weeks: 12
  backtest_window_weeks: 4
  ci_level_low: 0.80                 # 80% confidence interval
  ci_level_high: 0.95                # 95% confidence interval
  # auto-ARIMA search space
  arima_max_p: 5
  arima_max_d: 2
  arima_max_q: 5

# ── Bayesian Geo-Risk (W8) ────────────────────────────────────────────────────
georisk:
  prior_alpha: 1.0                   # Beta distribution prior α
  prior_beta: 5.0                    # Beta distribution prior β
  lambda_adjustment: 0.5             # h_adj = h_base * (1 + geo_risk * lambda)

# ── Newsvendor Model (W8) ─────────────────────────────────────────────────────
newsvendor:
  service_level: 0.95               # target CSL for non-Vital; Vital uses dynamic CR

# ── Game Theory / Supplier Qualification (W9-W10) ────────────────────────────
game_theory:
  dt_max_depth: 5
  dt_criterion: "gini"              # 'gini' or 'entropy'
  nash_iterations: 1000
"""

params_path = ROOT / "params.yaml"
params_path.write_text(PARAMS_YAML)

import yaml
with open(params_path) as f:
    params = yaml.safe_load(f)

# Validate weights sum to 1
weights = [params["classifier"][f"w{i}"] for i in range(1, 5)]
assert abs(sum(weights) - 1.0) < 1e-9, f"Classifier weights must sum to 1.0, got {sum(weights)}"

print(f"✅ params.yaml written ({len(PARAMS_YAML.splitlines())} lines)")
print(f"   Classifier weights: {weights} → sum={sum(weights):.2f} ✅")
print(f"   Sections: {list(params.keys())}")

---
## STEP 8 — Write `src/utils/data_io.py`

Typed, DVC-aware I/O helpers used by every downstream notebook and script.  
Centralises read/write logic so path changes propagate everywhere automatically.

In [ ]:
DATA_IO_PY = '''\
"""
src/utils/data_io.py
--------------------
Typed load/save helpers for the Geo-Aware MRO data pipeline.
All functions are DVC-aware: they resolve paths relative to project root
and validate that DVC-tracked files exist before reading.
"""
from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any

import pandas as pd
import duckdb

from src.config.settings import settings

logger = logging.getLogger(__name__)

ROOT = settings.PROJECT_ROOT


# ── Path helpers ──────────────────────────────────────────────────────────────

def raw(filename: str) -> Path:
    """Resolve a filename in data/raw/."""
    return ROOT / "data" / "raw" / filename


def processed(filename: str) -> Path:
    """Resolve a filename in data/processed/."""
    return ROOT / "data" / "processed" / filename


def external(filename: str) -> Path:
    """Resolve a filename in data/external/."""
    return ROOT / "data" / "external" / filename


def interim(filename: str) -> Path:
    """Resolve a filename in data/interim/."""
    return ROOT / "data" / "interim" / filename


# ── Generic loaders ──────────────────────────────────────────────────────────

def load_parquet(path: Path | str, **kwargs: Any) -> pd.DataFrame:
    """Load a parquet file with validation."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Parquet not found: {p}\n"
            "Run `dvc repro` to regenerate it, or check your DVC remote."
        )
    df = pd.read_parquet(p, **kwargs)
    logger.info("Loaded %s — shape=%s", p.name, df.shape)
    return df


def save_parquet(df: pd.DataFrame, path: Path | str, **kwargs: Any) -> None:
    """Save a DataFrame as parquet, creating parent dirs if needed."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(p, index=False, **kwargs)
    logger.info("Saved  %s — shape=%s", p.name, df.shape)


def load_csv(path: Path | str, **kwargs: Any) -> pd.DataFrame:
    """Load a CSV with validation."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p}")
    df = pd.read_csv(p, **kwargs)
    logger.info("Loaded %s — shape=%s", p.name, df.shape)
    return df


def save_json(obj: Any, path: Path | str, indent: int = 2) -> None:
    """Save a JSON-serialisable object."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=indent, default=str))
    logger.info("Saved  %s", p.name)


def load_json(path: Path | str) -> Any:
    """Load a JSON file."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"JSON not found: {p}")
    return json.loads(p.read_text())


# ── DuckDB helpers ────────────────────────────────────────────────────────────

def get_duckdb_conn(db_path: Path | str | None = None) -> duckdb.DuckDBPyConnection:
    """
    Return a DuckDB connection.
    If db_path is None, uses settings.DUCKDB_PATH (persistent).
    Pass ':memory:' for an in-memory connection.
    """
    path = str(db_path) if db_path is not None else str(settings.DUCKDB_PATH)
    conn = duckdb.connect(path)
    logger.info("DuckDB connection opened: %s", path)
    return conn


def parquet_to_duckdb(
    parquet_path: Path | str,
    table_name: str,
    conn: duckdb.DuckDBPyConnection | None = None,
    replace: bool = True,
) -> duckdb.DuckDBPyConnection:
    """Load a parquet file into a DuckDB table."""
    _conn = conn or get_duckdb_conn()
    qualifier = "OR REPLACE" if replace else "IF NOT EXISTS"
    _conn.execute(
        f"CREATE {qualifier} TABLE {table_name} AS "
        f"SELECT * FROM read_parquet('{parquet_path}')"
    )
    count = _conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    logger.info("DuckDB table '%s' loaded — %d rows", table_name, count)
    return _conn


# ── DVC pipeline metadata ────────────────────────────────────────────────────

def assert_dvc_output_exists(path: Path | str, stage_name: str) -> None:
    """
    Assert a DVC stage output exists, printing a helpful message if not.
    Call at the top of any notebook that depends on a prior DVC stage.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Expected output of DVC stage '{stage_name}' not found: {p}\n"
            f"Run: dvc repro {stage_name}"
        )
'''

data_io_path = ROOT / "src" / "utils" / "data_io.py"
data_io_path.parent.mkdir(parents=True, exist_ok=True)
data_io_path.write_text(DATA_IO_PY)
print(f"✅ src/utils/data_io.py written ({len(DATA_IO_PY.splitlines())} lines)")

# Quick import smoke test
import importlib.util
spec = importlib.util.spec_from_file_location("data_io", data_io_path)
# Not actually loading to avoid duckdb dependency check here — just confirm file exists
print(f"   File exists    : {data_io_path.exists()} ✅")

---
## STEP 9 — `dvc status` check

Verify the DVC config is clean before committing.  
`dvc status` should show no tracked files yet (we haven't added any data),  
but the pipeline stages from `dvc.yaml` should be listed as `never run`.

In [ ]:
# ── dvc status ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ["dvc", "status"],
    cwd=ROOT,
    capture_output=True,
    text=True
)
print("── dvc status output ──")
print(result.stdout or "(no output — all stages fresh or no data tracked yet)")
if result.stderr:
    print("STDERR:", result.stderr[:500])

# ── dvc dag ───────────────────────────────────────────────────────────────────
dag_result = subprocess.run(
    ["dvc", "dag"],
    cwd=ROOT,
    capture_output=True,
    text=True
)
print("\n── dvc dag ──")
print(dag_result.stdout or "(dag output empty — data not yet tracked)")

# ── dvc remote list ───────────────────────────────────────────────────────────
print("\n── dvc remote list ──")
print(run_dvc("remote", "list"))

---
## STEP 10 — Log DVC setup metadata to MLflow

In [ ]:
import mlflow
import yaml

mlflow.set_tracking_uri(settings.MLFLOW_URI)
mlflow.set_experiment("W1_Infrastructure")

# Read params for logging
with open(ROOT / "params.yaml") as f:
    params = yaml.safe_load(f)

with mlflow.start_run(run_name="W1D3_dvc_setup") as run:
    # DVC setup metadata
    mlflow.log_param("dvc_version",        dvc_version)
    mlflow.log_param("dvc_default_remote", run_dvc("remote", "default"))
    mlflow.log_param("pipeline_stages",    len(parsed["stages"]))
    mlflow.log_param("data_layers",        len(DATA_LAYER_METADATA))

    # Key pipeline params (for future comparison)
    mlflow.log_param("classifier_w1",      params["classifier"]["w1"])
    mlflow.log_param("classifier_w2",      params["classifier"]["w2"])
    mlflow.log_param("classifier_w3",      params["classifier"]["w3"])
    mlflow.log_param("classifier_w4",      params["classifier"]["w4"])
    mlflow.log_param("n_skus_target",      params["sku_master"]["n_skus"])
    mlflow.log_param("adi_threshold",      params["classifier"]["adi_threshold"])
    mlflow.log_param("cv2_threshold",      params["classifier"]["cv2_threshold"])

    # Log dvc.yaml as artifact for traceability
    mlflow.log_artifact(str(ROOT / "dvc.yaml"),    artifact_path="pipeline")
    mlflow.log_artifact(str(ROOT / "params.yaml"), artifact_path="pipeline")

    mlflow.set_tag("day",    "W1D3")
    mlflow.set_tag("status", "dvc_setup_complete")

print(f"✅ MLflow run logged: {run.info.run_id}")
print(f"   Experiment     : W1_Infrastructure")
print(f"   Artifacts      : dvc.yaml + params.yaml")

---
## STEP 11 — Git commit: W1D3

Commit all Day 3 artefacts to the `develop` branch.

**Files changed today:**
```
.dvc/config                  ← DVC remote config
.dvc/.gitignore              ← auto-generated
.dvcignore                   ← DVC ignore patterns
dvc.yaml                     ← 9-stage pipeline DAG
params.yaml                  ← all hyperparameters
data/raw/README.md
data/raw/schema.json
data/processed/README.md
data/processed/schema.json
data/external/README.md
data/external/schema.json
data/interim/README.md
data/interim/schema.json
src/utils/data_io.py
```

In [ ]:
GIT_COMMANDS = """
# ── W1D3 Git commit sequence ─────────────────────────────────────────────────
# Run these in terminal from the project root:

cd ~/geo-aware-mro
git checkout develop

# Stage DVC init files
git add .dvc/.gitignore
git add .dvc/config
git add .dvcignore

# Stage pipeline definition files
git add dvc.yaml
git add params.yaml

# Stage data layer metadata (NOT the actual data files)
git add data/raw/README.md data/raw/schema.json data/raw/.gitkeep
git add data/processed/README.md data/processed/schema.json data/processed/.gitkeep
git add data/external/README.md data/external/schema.json data/external/.gitkeep
git add data/interim/README.md data/interim/schema.json data/interim/.gitkeep

# Stage new source files
git add src/utils/data_io.py

# Commit
git commit -m 'chore: W1D3 — DVC init, remote config, pipeline DAG, params, data_io'

# Push to remote
git push origin develop
"""
print(GIT_COMMANDS)

---
## ✅ Day 3 Checklist

| Task | Done? |
|------|-------|
| `dvc --version` ≥ 3.x confirmed | ☐ |
| `dvc init` complete (`.dvc/` exists) | ☐ |
| `localremote` added as default DVC remote | ☐ |
| `.dvcignore` written | ☐ |
| `dvc.yaml` — 9 stages defined | ☐ |
| `params.yaml` — all hyperparams set, weights sum to 1.0 | ☐ |
| `data/raw/` — README.md + schema.json seeded | ☐ |
| `data/processed/` — README.md + schema.json seeded | ☐ |
| `data/external/` — README.md + schema.json seeded | ☐ |
| `data/interim/` — README.md + schema.json seeded | ☐ |
| `src/utils/data_io.py` written (typed helpers) | ☐ |
| MLflow W1D3 run logged with `dvc.yaml` + `params.yaml` artifacts | ☐ |
| `dvc status` clean (no errors) | ☐ |
| Git commit on `develop`: `W1D3 — DVC init...` | ☐ |
| Pushed to GitHub | ☐ |

---

## 🔜 Day 4 (Thu) Preview

**Dockerfile + `docker-compose.yml` + GitHub Actions CI pipeline**

- Multi-stage `Dockerfile`: builder → runtime image with `geo-mro` env
- `docker-compose.yml`: services for app + MLflow server + DuckDB volume
- `.github/workflows/ci.yml`: lint (ruff) + test (pytest) + docker build
- Badge in README: `![CI](https://github.com/.../actions/...badge.svg)`
- Verify: `docker-compose up` launches all three services